# S6E9 — Kaggle-ready 0.94644 Micro-Blend

This version is designed to run from a clean Kaggle session with **Run All**.

### Add these inputs before running

1. Competition data: **Predicting Electric Vehicle Purchases** (`playground-series-s6e9`).
2. Dataset: **S6E9 Zoom Zoom Baseline** by `jazivxt` (`jazivxt/s6e9-zoom-zoom-baseline`).

The notebook needs no internet, performs no training and does not run Optuna. It reads only four selected prediction files directly from the mounted collection and writes three valid submissions to `/kaggle/working`. The requested 0.94644 is a target, not a guaranteed or locally measured score.

## Sources and acknowledgements

- [Single Model — Zoom Zoom](https://www.kaggle.com/code/jazivxt/single-model-zoom-zoom) by **jazivxt** — exact scored 0.94643 reference and prediction collection.
- [S6E9 LB 0.94643 And Six Missing Sources](https://www.kaggle.com/code/megayak/s6e9-lb-0-94643-and-six-missing-sources) by **megayak** — independent public 0.94643 prediction.
- [S6E9 | 94.6+ | Transformer and GBDT Ensemble](https://www.kaggle.com/code/lamhuy8904/s6e9-94-6-transformer-and-gbdt-ensemble) by **Lâm Huy** — diversity component.
- [Pure LGBM Model CV 0.94606 LB 0.94637](https://www.kaggle.com/code/najiama/pure-lgbm-model-cv-0-94606-lb-0-94637) by **Naji Ama** and [Simple 21-feature LGBM](https://www.kaggle.com/code/sergeyqt2024/simple-21-feature-lfbm-0-9456) by **Sergey** — foundations used in the preceding experiments.

Thank you to the authors for sharing their work. Please upvote the original notebooks if these predictions help.

In [ ]:
from io import BytesIO
from pathlib import Path
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import norm, rankdata

TARGET = 'Will_Buy_EV'
ID = 'id'
KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

def find_unique(filename):
    roots = [KAGGLE_INPUT] if KAGGLE_INPUT.is_dir() else [Path.cwd()]
    matches = []
    for root in roots:
        direct = root / filename
        if direct.is_file():
            matches.append(direct)
        matches.extend(root.glob('**/' + filename))
    matches = list(dict.fromkeys(path.resolve() for path in matches if path.is_file()))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one {filename}, found {len(matches)}. '
            'Check that the competition and jazivxt/s6e9-zoom-zoom-baseline are attached. '
            f'Matches: {matches[:5]}'
        )
    return matches[0]

sample_path = find_unique('sample_submission.csv')
reference_path = find_unique('submission_latest_best.csv')
archive_path = find_unique('ranked_predictions_latest.zip.bin')

print('Competition sample:', sample_path)
print('0.94643 reference:', reference_path)
print('Prediction archive:', archive_path)
print('Output directory:', WORKING)

## Read only the required predictions

The collection contains hundreds of CSV files. Opening the ZIP directly avoids extracting roughly a gigabyte into the limited Kaggle working directory. Exact SHA-256 checks prevent silently blending a changed or incorrectly attached file.

In [ ]:
EXPECTED = {
    'reference_94643_a': '416ebd9d73c6464d2c55c12a34846eddb6687f2dec808c4111b9681ce59cb4e7',
    'reference_94643_b': '683d761ca77fdd5bc07a0fb1b62678d80c927111cebbfd8c9b6778ce31c8a5e6',
    'megayak_94643': '93f2d0ff1e3f2284769a772e61b04c171b19dfe5732d1ff2548715d8f3188ac8',
    'transformer_94608': 'f11c5d3b6057436b802360a054eadac974792e5d2b9ab8d0acb9878258ee5969',
}

ARCHIVE_MEMBERS = {
    'reference_94643_b': 'own/own_002_56112861.csv',
    'megayak_94643': (
        'public/megayak__s6e9-lb-0-94643-and-six-missing-sources/'
        'v348386264_93f2d0ff1e3f.csv'
    ),
    'transformer_94608': (
        'public/lamhuy8904__s6e9-94-6-transformer-and-gbdt-ensemble/'
        'v346834462_f11c5d3b6057.csv'
    ),
}

def checked_frame_from_bytes(name, raw):
    digest = hashlib.sha256(raw).hexdigest()
    assert digest == EXPECTED[name], f'Hash mismatch for {name}: {digest}'
    frame = pd.read_csv(BytesIO(raw))
    assert list(frame.columns) == [ID, TARGET]
    assert frame[ID].is_unique
    assert np.isfinite(frame[TARGET]).all()
    return frame, digest

reference_raw = reference_path.read_bytes()
reference_frame, reference_hash = checked_frame_from_bytes(
    'reference_94643_a', reference_raw
)

frames = {'reference_94643_a': reference_frame}
hashes = {'reference_94643_a': reference_hash}
with zipfile.ZipFile(archive_path) as archive:
    available = set(archive.namelist())
    missing = set(ARCHIVE_MEMBERS.values()) - available
    assert not missing, f'Missing archive members: {sorted(missing)}'
    for name, member in ARCHIVE_MEMBERS.items():
        raw = archive.read(member)
        frames[name], hashes[name] = checked_frame_from_bytes(name, raw)

sample = pd.read_csv(sample_path)
assert sample[ID].is_unique
for name, frame in frames.items():
    assert frame[ID].equals(sample[ID]), f'ID order mismatch: {name}'

print(f'Loaded and verified {len(frames)} prediction files for {len(sample):,} test rows.')

## Diversity-aware weight

ROC-AUC depends on ordering, so all sources are converted to percentile ranks. The Transformer+GBDT prediction is weaker by itself but meaningfully less correlated with the 0.94643 reference. A small contribution can therefore improve ordering without replacing the strong reference.

The weight is derived from reported AUC values and observed rank correlation using a simple binormal reliability approximation. It is not optimized against hidden Kaggle labels.

In [ ]:
def unit_rank(values):
    return (rankdata(values, method='average') - .5) / len(values)

ranks = pd.DataFrame({
    name: unit_rank(frame[TARGET].to_numpy(float))
    for name, frame in frames.items()
})

correlation = ranks.corr()
rho = correlation.loc['reference_94643_a', 'transformer_94608']
auc_reference, auc_transformer = .94643, .94608
d_reference, d_transformer = np.sqrt(2) * norm.ppf([auc_reference, auc_transformer])

def approximate_separation(transformer_weight):
    w = float(transformer_weight)
    numerator = (1 - w) * d_reference + w * d_transformer
    variance = (1 - w) ** 2 + w ** 2 + 2 * rho * w * (1 - w)
    return numerator / np.sqrt(variance)

result = minimize_scalar(
    lambda w: -approximate_separation(w),
    bounds=(0, .15), method='bounded', options={'xatol': 1e-12}
)
derived_weight = float(result.x)

display(pd.DataFrame({
    'reported_public_auc': {
        'reference_94643_a': .94643,
        'reference_94643_b': .94643,
        'megayak_94643': .94643,
        'transformer_94608': .94608,
    },
    'rank_correlation_to_reference': correlation['reference_94643_a'],
}).round(9))
print(f'Derived Transformer weight: {derived_weight:.4%}')
print('Locked primary weight: 5.5%')

## Create Kaggle submissions

Recommended submission order:

1. `submission_ref43_transformer055.csv` — primary 5.5% diversity blend.
2. `submission_consensus43_transformer050.csv` — three-reference consensus with 5% Transformer.
3. `submission_ref43_transformer030.csv` — conservative 3% alternative.

Submit the first file initially. The other two are controlled fallbacks, not claims of higher scores.

In [ ]:
reference = ranks['reference_94643_a'].to_numpy()
transformer = ranks['transformer_94608'].to_numpy()
consensus_43 = (
    .50 * ranks['reference_94643_a'].to_numpy() +
    .25 * ranks['reference_94643_b'].to_numpy() +
    .25 * ranks['megayak_94643'].to_numpy()
)

candidates = {
    'submission_ref43_transformer055.csv': .945 * reference + .055 * transformer,
    'submission_consensus43_transformer050.csv': .95 * consensus_43 + .05 * transformer,
    'submission_ref43_transformer030.csv': .97 * reference + .03 * transformer,
}

qa = []
for filename, prediction in candidates.items():
    submission = sample.copy()
    submission[TARGET] = prediction
    assert len(submission) == len(sample)
    assert submission[ID].equals(sample[ID]) and submission[ID].is_unique
    assert np.isfinite(submission[TARGET]).all()
    assert submission[TARGET].between(0, 1).all()
    output_path = WORKING / filename
    submission.to_csv(output_path, index=False)
    qa.append({
        'file': filename,
        'rows': len(submission),
        'min': float(prediction.min()),
        'max': float(prediction.max()),
        'sha256': hashlib.sha256(output_path.read_bytes()).hexdigest(),
    })

qa_frame = pd.DataFrame(qa)
display(qa_frame)
print('Primary submission:', WORKING / 'submission_ref43_transformer055.csv')

## Final audit

The report records exact source hashes, weights and output names. Public scores are author/user-reported historical observations. The generated candidates remain unmeasured until Kaggle evaluates them. Since the public leaderboard uses only a subset of test rows, do not treat a 0.00001 movement as conclusive evidence for the private leaderboard.

In [ ]:
report = {
    'target_public_auc': .94644,
    'reference_reported_public_auc': .94643,
    'transformer_reported_public_auc': .94608,
    'reference_transformer_rank_correlation': float(rho),
    'derived_transformer_weight': derived_weight,
    'primary_transformer_weight': .055,
    'recommended_submission_order': list(candidates),
    'source_hashes': hashes,
    'output_hashes': dict(zip(qa_frame['file'], qa_frame['sha256'])),
    'score_status': 'Unmeasured until submitted to Kaggle',
}
report_path = WORKING / 'target94644_kaggle_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))